In [1]:
import pandas as pd
import os
import numpy as np
import re

In [2]:
pd.set_option('display.max_columns', None)

# Data curation

## **STEP 1**. Merge dicomtocsv_series.csv and dicomtocsv_study.csv

### <span style="color:darkcyan">**Function**</span>

In [3]:
def merge_and_extract(file_path, file_series, file_study):
    """
    Merges two files into a single file based on shared columns, then extract desired columns
    """       
    df_dicom_series = pd.read_csv(os.path.join(file_path, file_series)).dropna(axis=1, how='all')
    df_dicom_study = pd.read_csv(os.path.join(file_path, file_study)).dropna(axis=1, how='all')

    series_cols = df_dicom_series.columns
    study_cols = df_dicom_study.columns

    common_cols = series_cols.intersection(study_cols)
    series_not_study = series_cols.difference(study_cols)
    study_not_series = study_cols.difference(series_cols)

    df_merge = pd.merge(
    df_dicom_series,
    df_dicom_study,
    how="left",
    on=list(common_cols)
    )
    
    return df_merge, df_dicom_series, df_dicom_study

### <span style="color:blue">**Main**</span>

In [4]:
file_path = "P:/Dataset/R3Data"
file_series = "dicomtocsv_series.csv"
file_study = "dicomtocsv_study.csv"

df_merge, df_dicom_series, df_dicom_study = merge_and_extract(file_path, file_series, file_study)

In [5]:
df_merge.shape, df_dicom_series.shape, df_dicom_study.shape

((136233, 39), (136199, 36), (15561, 14))

In [6]:
df_merge.columns

Index(['NumberOfReferences', 'ImageType', 'SOPClassUID', 'StudyDate',
       'AcquisitionDate', 'AcquisitionDateTime', 'StudyTime',
       'AccessionNumber', 'Modality', 'Manufacturer', 'StudyDescription',
       'SeriesDescription', 'ManufacturerModelName', 'PatientID',
       'PatientBirthDate', 'PatientSex', 'ScanningSequence', 'SequenceVariant',
       'ScanOptions', 'MRAcquisitionType', 'SequenceName', 'SliceThickness',
       'MagneticFieldStrength', 'SpacingBetweenSlices', 'Exposure',
       'StudyInstanceUID', 'SeriesInstanceUID', 'StudyID', 'SeriesNumber',
       'InstanceNumber', 'NumberOfFrames', 'Rows', 'Columns', 'PixelSpacing',
       'WindowCenterWidthExplanation', 'TransferSyntaxUID',
       'PerformingPhysicianName', 'PatientAge', 'BodyPartExamined'],
      dtype='object')

In [7]:
df_all = df_merge.copy()

In [8]:
extract_cols = ['PatientID', 'PatientSex', 'PatientBirthDate', 'PatientAge',
            'AccessionNumber',
            'Manufacturer', 'ManufacturerModelName', 'Modality',
            'StudyDate', 'AcquisitionDate', 'StudyTime', 'StudyDescription', 'StudyInstanceUID', 'StudyID',
            'SeriesDescription', 'SeriesNumber', 'SeriesInstanceUID', 'InstanceNumber',
            'ImageType',
            'SliceThickness', 'SpacingBetweenSlices', 'Exposure', 'NumberOfFrames', 'Rows', 'Columns', 'PixelSpacing'
]


In [9]:
df_focus = df_all[extract_cols]

In [10]:
df_focus.head(5)

,PatientID,PatientSex,PatientBirthDate,PatientAge,AccessionNumber,Manufacturer,ManufacturerModelName,Modality,StudyDate,AcquisitionDate,StudyTime,StudyDescription,StudyInstanceUID,StudyID,SeriesDescription,SeriesNumber,SeriesInstanceUID,InstanceNumber,ImageType,SliceThickness,SpacingBetweenSlices,Exposure,NumberOfFrames,Rows,Columns,PixelSpacing
0,4333469193,F,1964-07-01,050Y,80861023,"HOLOGIC, Inc.",Selenia Dimensions,MG,2015-01-02,2015-01-02,08:39:23,BREAST IMAGING TOMOSYNTHESIS DIAGNOSTIC BILATERAL,1.400756,1.113140,L ISO,71100000.0,1.882280,367.0,DERIVED\PRIMARY,NaN,NaN,27.0,NaN,4096.0,3328.0,0.065238\0.065238
1,4333843993,F,1967-07-01,047Y,81776067,"R2 Technology, Inc.",Cenova,MG,2015-01-02,NaN,11:30:52,DIAG MAMMO - DIRECT DIG IMAGE BIL ALL VIEWS,1.233100,1.119129,R2 CAD SC,1.0,1.347473,1.0,DERIVED\SECONDARY,NaN,NaN,NaN,NaN,1500.0,1250.0,NaN
2,4333843993,F,1967-07-01,NaN,81776067,"HOLOGIC, Inc.",Selenia Dimensions,MG,2015-01-02,2015-01-02,11:30:52,MAMMOGRAM DIGITAL DX BILAT,1.233100,1.119129,L CC,71300000.0,1.936507,87.0,DERIVED\PRIMARY,NaN,NaN,246.0,NaN,4096.0,3328.0,0.065238\0.065238
3,4333843993,F,1967-07-01,NaN,81776067,"HOLOGIC, Inc.",Selenia Dimensions,MG,2015-01-02,2015-01-02,11:30:52,MAMMOGRAM DIGITAL DX BILAT,1.233100,1.119129,L MLO,71300000.0,1.220605,93.0,DERIVED\PRIMARY,NaN,NaN,250.0,NaN,4096.0,3328.0,0.065238\0.065238
4,4333843993,F,1967-07-01,NaN,81776067,"HOLOGIC, Inc.",Selenia Dimensions,MG,2015-01-02,2015-01-02,11:30:52,MAMMOGRAM DIGITAL DX BILAT,1.233100,1.119129,R CC,71300000.0,1.340178,75.0,DERIVED\PRIMARY,NaN,NaN,197.0,NaN,4096.0,3328.0,0.065238\0.065238


## **STEP 2**. Group by PatientID and calculate Age

### <span style="color:darkcyan">**Function**</span>

In [11]:
def calculate_age_at_study(df):
    """
    Calculates the patient's age in years at the time of the study 
    based on the 'PatientBirthDate' and 'StudyDate' columns.
    """
    
    # 1. Ensure date columns are in datetime format
    # The format 'YYYY-MM-DD' is used for parsing
    try:
        df['BIRTH_DATE_DT'] = pd.to_datetime(df['BIRTH_DATE'], format='%Y-%m-%d')
        # Note: The StudyDate column often includes time (e.g., '2020-06-01 09:10:52').
        # We can let pandas infer the format for this one since it's cleaner.
        df['StudyDate_DT'] = pd.to_datetime(df['StudyDate'], errors='coerce') 
    except ValueError as e:
        print(f"Error parsing date format: {e}. Please check your date column formats.")
        return df

    # 2. Calculate the difference in days
    time_difference = df['StudyDate_DT'] - df['BIRTH_DATE_DT']
    
    # 3. Convert the difference into whole years (integer format)
    # The .dt.days attribute gives the number of days, which is divided by 365.25 
    # and then explicitly cast to an integer to capture only the full years elapsed.
    df['PatientAge'] = (time_difference.dt.days / 365.25).astype(int)
    
    # Clean up the intermediate columns
    df = df.drop(columns=['BIRTH_DATE_DT', 'StudyDate_DT'])
    
    return df

### <span style="color:blue">**Main**</span>

In [12]:
df_sort = df_focus.sort_values(
        by=['PatientID', 'StudyDate', 'AccessionNumber'],
        ignore_index=True
    )

num_patient = df_sort["PatientID"].nunique()
print(num_patient)

5515


In [13]:
df_sort.rename(columns={'PatientID': 'PATIENT_STUDY_ID', 'AccessionNumber': 'ACCESSION_NUMBER', 'PatientBirthDate': 'BIRTH_DATE'}, inplace=True)

In [14]:
df_step2 = calculate_age_at_study(df_sort)

In [15]:
df_step2[df_step2['PATIENT_STUDY_ID']==4335033113].head(5)

,PATIENT_STUDY_ID,PatientSex,BIRTH_DATE,PatientAge,ACCESSION_NUMBER,Manufacturer,ManufacturerModelName,Modality,StudyDate,AcquisitionDate,StudyTime,StudyDescription,StudyInstanceUID,StudyID,SeriesDescription,SeriesNumber,SeriesInstanceUID,InstanceNumber,ImageType,SliceThickness,SpacingBetweenSlices,Exposure,NumberOfFrames,Rows,Columns,PixelSpacing
120554,4335033113,F,1959-07-01,60,60229362,"R2 Technology, Inc.",Cenova,MG,2020-03-13,NaN,11:57:21,MAMMOGRAM DIGITAL SCR BILAT,1.238332,1.786414,Hologic R2 ImageChecker CAD SC,1.0,1.335281,1.0,DERIVED\SECONDARY,NaN,NaN,NaN,NaN,1500.0,1250.0,NaN
120555,4335033113,F,1959-07-01,60,60229362,"HOLOGIC, Inc.",Selenia Dimensions,MG,2020-03-13,NaN,11:57:21,MAMMOGRAM DIGITAL SCR BILAT,1.238332,1.786414,L MLO Intelligent 2D,71300000.0,1.126898,223.0,DERIVED\PRIMARY\TOMOSYNTHESIS\GENERATED_2D,NaN,NaN,80.0,NaN,4096.0,3328.0,0.063901\0.063901
120556,4335033113,F,1959-07-01,60,60229362,"HOLOGIC, Inc.",Selenia Dimensions,MG,2020-03-13,NaN,11:57:21,MAMMOGRAM DIGITAL SCR BILAT,1.238332,1.786414,R MLO Intelligent 2D,71300000.0,1.145594,217.0,DERIVED\PRIMARY\TOMOSYNTHESIS\GENERATED_2D,NaN,NaN,74.0,NaN,4096.0,3328.0,0.064002\0.064002
120557,4335033113,F,1959-07-01,60,60229362,"HOLOGIC, Inc.",Selenia Dimensions,MG,2020-03-13,NaN,11:57:21,MAMMOGRAM DIGITAL SCR BILAT,1.238332,1.786414,R MLO Intelligent 2D,71300000.0,1.205296,211.0,DERIVED\PRIMARY\TOMOSYNTHESIS\GENERATED_2D,NaN,NaN,77.0,NaN,4096.0,3328.0,0.063901\0.063901
120558,4335033113,F,1959-07-01,60,60229362,"HOLOGIC, Inc.",Selenia Dimensions,MG,2020-03-13,NaN,11:57:21,MAMMOGRAM DIGITAL SCR BILAT,1.238332,1.786414,R CC Intelligent 2D,71300000.0,1.209709,205.0,DERIVED\PRIMARY\TOMOSYNTHESIS\GENERATED_2D,NaN,NaN,65.0,NaN,4096.0,3328.0,0.064302\0.064302


## **STEP 3.** Add tags (Study, Side, Series)

In [16]:
dicom = df_step2

In [17]:
dicom.head(3)

,PATIENT_STUDY_ID,PatientSex,BIRTH_DATE,PatientAge,ACCESSION_NUMBER,Manufacturer,ManufacturerModelName,Modality,StudyDate,AcquisitionDate,StudyTime,StudyDescription,StudyInstanceUID,StudyID,SeriesDescription,SeriesNumber,SeriesInstanceUID,InstanceNumber,ImageType,SliceThickness,SpacingBetweenSlices,Exposure,NumberOfFrames,Rows,Columns,PixelSpacing
0,4330018595,F,1965-07-01,54,63737104,"R2 Technology, Inc.",Cenova,MG,2019-08-19,NaN,02:11:20,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.267362,1.556582,Hologic R2 ImageChecker CAD SC,1.0,1.217865,1.0,DERIVED\SECONDARY,NaN,NaN,NaN,NaN,1500.0,1250.0,NaN
1,4330018595,F,1965-07-01,54,63737104,"HOLOGIC, Inc.",Selenia Dimensions,MG,2019-08-19,2019-08-19,02:11:20,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.267362,1.556582,R ML,71100000.0,1.132071,314.0,DERIVED\PRIMARY,NaN,NaN,102.0,NaN,3328.0,2560.0,0.038889\0.038889
2,4330018595,F,1965-07-01,54,63737104,"HOLOGIC, Inc.",Selenia Dimensions,MG,2019-08-19,2019-08-19,02:11:20,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.267362,1.556582,R XCCL,71100000.0,1.186464,312.0,DERIVED\PRIMARY,NaN,NaN,94.0,NaN,3328.0,2560.0,0.038889\0.038889


In [18]:
dicom["Modality"].unique()

array(['MG', 'PR', 'SR', 'US', 'MR', 'OT', 'KO', 'SC'], dtype=object)

In [19]:
laterality_bi = ["BILATERAL", "BILAT", "BIL", "BI"]
laterality_uni = ["UNILATERAL", "UNILAT", "UNI", "Unilateral"]
other = ["IMPLANT", "CONTRAST ENHANCED", "Implant"]

In [20]:
dicom_copy = dicom.copy()

### <span style="color:blue"> **Study**</span> (SCREEN, DIAG)

In [21]:
study_types = {
    "DIAG":   ["DIAG", "DIAGNOSTIC", "DX"],
    "SCREEN": ["SCREENING", "SCREEN"],
}

In [22]:
column_to_check = 'StudyDescription'
type_column = 'study'

dicom_copy[column_to_check] = dicom_copy[column_to_check].astype(str)

for label, terms in study_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = dicom_copy[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    dicom_copy.loc[mask, type_column] = label

### <span style="color:blue">**Side**</span> (R, L)

In [23]:
side_types = {
    "R": ["RIGHT", "RT", "R XCCL", "R MLO", "R CC", "R ML", "R SIO", "R LM"],
    "L": ["LEFT", "LT", "L XCCL", "L MLO", "L CC", "L ML", "L SIO", "L LM"],
}

In [24]:
column_to_check = 'SeriesDescription'
type_column = 'side'

dicom_copy[column_to_check] = dicom_copy[column_to_check].astype(str)

for label, terms in side_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = dicom_copy[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    dicom_copy.loc[mask, type_column] = label

### <span style="color:blue">**Series**</span> (DBT, IN2D, C VIEW, SECURE)

In [25]:
series_types = {
    "DBT":    ["Breast Tomosynthesis"],
    "IN2D":   ["Intelligent 2D"],
    "C VIEW": ["C-View"],
    "SECURE": ["SecurView"],
}

In [26]:
column_to_check = 'SeriesDescription'
type_column = 'series'

dicom_copy[column_to_check] = dicom_copy[column_to_check].astype(str)

for label, terms in series_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = dicom_copy[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    dicom_copy.loc[mask, type_column] = label

### <span style="color:blue">**View**</span> (MLO, CC)

In [27]:
view_types = {
    "MLO": ["L MLO", "R MLO"],
    "CC":  ["L CC", "R CC"],
}

In [28]:
column_to_check = 'SeriesDescription'
type_column = 'view'

dicom_copy[column_to_check] = dicom_copy[column_to_check].astype(str)

for label, terms in view_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = dicom_copy[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    dicom_copy.loc[mask, type_column] = label

### <span style="color:blue">**Reorder columns**</span>

In [29]:
dicom_copy.columns

Index(['PATIENT_STUDY_ID', 'PatientSex', 'BIRTH_DATE', 'PatientAge',
       'ACCESSION_NUMBER', 'Manufacturer', 'ManufacturerModelName', 'Modality',
       'StudyDate', 'AcquisitionDate', 'StudyTime', 'StudyDescription',
       'StudyInstanceUID', 'StudyID', 'SeriesDescription', 'SeriesNumber',
       'SeriesInstanceUID', 'InstanceNumber', 'ImageType', 'SliceThickness',
       'SpacingBetweenSlices', 'Exposure', 'NumberOfFrames', 'Rows', 'Columns',
       'PixelSpacing', 'study', 'side', 'series', 'view'],
      dtype='object')

In [30]:
dicom_copy = dicom_copy[['PATIENT_STUDY_ID', 'PatientSex', 'BIRTH_DATE', 'PatientAge',
       'ACCESSION_NUMBER', 'study', 'side', 'series', 'view',
       'Manufacturer', 'ManufacturerModelName', 'Modality',
       'StudyDate', 'AcquisitionDate', 'StudyTime', 'StudyDescription',
       'StudyInstanceUID', 'StudyID', 'SeriesDescription', 'SeriesNumber',
       'SeriesInstanceUID', 'InstanceNumber', 'ImageType', 'SliceThickness',
       'SpacingBetweenSlices', 'Exposure', 'NumberOfFrames', 'Rows', 'Columns',
       'PixelSpacing']]

In [31]:
dicom_copy.head(3)

,PATIENT_STUDY_ID,PatientSex,BIRTH_DATE,PatientAge,ACCESSION_NUMBER,study,side,series,view,Manufacturer,ManufacturerModelName,Modality,StudyDate,AcquisitionDate,StudyTime,StudyDescription,StudyInstanceUID,StudyID,SeriesDescription,SeriesNumber,SeriesInstanceUID,InstanceNumber,ImageType,SliceThickness,SpacingBetweenSlices,Exposure,NumberOfFrames,Rows,Columns,PixelSpacing
0,4330018595,F,1965-07-01,54,63737104,DIAG,NaN,NaN,NaN,"R2 Technology, Inc.",Cenova,MG,2019-08-19,NaN,02:11:20,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.267362,1.556582,Hologic R2 ImageChecker CAD SC,1.0,1.217865,1.0,DERIVED\SECONDARY,NaN,NaN,NaN,NaN,1500.0,1250.0,NaN
1,4330018595,F,1965-07-01,54,63737104,DIAG,R,NaN,NaN,"HOLOGIC, Inc.",Selenia Dimensions,MG,2019-08-19,2019-08-19,02:11:20,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.267362,1.556582,R ML,71100000.0,1.132071,314.0,DERIVED\PRIMARY,NaN,NaN,102.0,NaN,3328.0,2560.0,0.038889\0.038889
2,4330018595,F,1965-07-01,54,63737104,DIAG,R,NaN,NaN,"HOLOGIC, Inc.",Selenia Dimensions,MG,2019-08-19,2019-08-19,02:11:20,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.267362,1.556582,R XCCL,71100000.0,1.186464,312.0,DERIVED\PRIMARY,NaN,NaN,94.0,NaN,3328.0,2560.0,0.038889\0.038889


### <span style="color:#FF6347;">**SAVE**</span> file

In [32]:
path = "O:/R01-MO-DBT/MO-DBT-data-curation/Data"

In [33]:
output_file = os.path.join(path,'dicom_tag' + ".xlsx")
dicom_copy.to_excel(output_file, index=False)

### <span style="color:#FF6347;">**READ**</span> file

In [34]:
file_path = os.path.join(path,'dicom_tag' + ".xlsx")
dicom = pd.read_excel(file_path)

In [35]:
dicom[dicom["series"]=="DBT"]

,PATIENT_STUDY_ID,PatientSex,BIRTH_DATE,PatientAge,ACCESSION_NUMBER,study,side,series,view,Manufacturer,ManufacturerModelName,Modality,StudyDate,AcquisitionDate,StudyTime,StudyDescription,StudyInstanceUID,StudyID,SeriesDescription,SeriesNumber,SeriesInstanceUID,InstanceNumber,ImageType,SliceThickness,SpacingBetweenSlices,Exposure,NumberOfFrames,Rows,Columns,PixelSpacing
48,4330066079,F,1989-07-01,30,61259016,DIAG,L,DBT,MLO,"HOLOGIC, Inc.",Selenia Dimensions,MG,2020-01-14,NaN,18:25:57,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.215642,1.198432,L MLO Breast Tomosynthesis Image,73200000.0,1.256929,213.0,DERIVED\PRIMARY\VOLUME\NONE,1.0,NaN,NaN,47.0,2457.0,1890.0,0.088500\0.088500
49,4330066079,F,1989-07-01,30,61259016,DIAG,L,DBT,CC,"HOLOGIC, Inc.",Selenia Dimensions,MG,2020-01-14,NaN,18:25:57,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.215642,1.198432,L CC Breast Tomosynthesis Image,73200000.0,1.338235,219.0,DERIVED\PRIMARY\VOLUME\NONE,1.0,NaN,NaN,49.0,2457.0,1890.0,0.088364\0.088364
50,4330066079,F,1989-07-01,30,61259016,DIAG,R,DBT,MLO,"HOLOGIC, Inc.",Selenia Dimensions,MG,2020-01-14,NaN,18:25:58,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.215642,1.198432,R MLO Breast Tomosynthesis Image,73200000.0,1.301539,207.0,DERIVED\PRIMARY\VOLUME\NONE,1.0,NaN,NaN,47.0,2457.0,1890.0,0.088500\0.088500
51,4330066079,F,1989-07-01,30,61259016,DIAG,R,DBT,CC,"HOLOGIC, Inc.",Selenia Dimensions,MG,2020-01-14,NaN,18:25:58,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.215642,1.198432,R CC Breast Tomosynthesis Image,73200000.0,1.185590,201.0,DERIVED\PRIMARY\VOLUME\NONE,1.0,NaN,NaN,46.0,2457.0,1890.0,0.088500\0.088500
97,4330116791,F,1963-07-01,56,61499674,DIAG,L,DBT,MLO,"HOLOGIC, Inc.",Selenia Dimensions,MG,2019-12-17,NaN,15:18:12,DIAG DIG MAMMO LEFT ALL VIEWS WITH TOMOSYNTHESIS,1.309920,1.337933,L MLO Breast Tomosynthesis Image,73200000.0,1.897383,314.0,DERIVED\PRIMARY\VOLUME\NONE,1.0,NaN,NaN,83.0,2457.0,1996.0,0.105903\0.105903
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135993,4339601084,F,1972-07-01,46,77038224,DIAG,L,DBT,CC,"HOLOGIC, Inc.",Selenia Dimensions,MG,2018-11-07,NaN,01:46:47,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.599610,1.618870,L CC Breast Tomosynthesis Image,73200000.0,1.183151,12.0,DERIVED\PRIMARY\TOMOSYNTHESIS\NONE,1.0,NaN,NaN,77.0,2457.0,1996.0,0.106525\0.106525
135994,4339601084,F,1972-07-01,46,77038224,DIAG,R,DBT,MLO,"HOLOGIC, Inc.",Selenia Dimensions,MG,2018-11-07,NaN,01:46:46,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.599610,1.618870,R MLO Breast Tomosynthesis Image,73200000.0,1.238866,24.0,DERIVED\PRIMARY\TOMOSYNTHESIS\NONE,1.0,NaN,NaN,73.0,2457.0,1996.0,0.106860\0.106860
135995,4339601084,F,1972-07-01,46,77038224,DIAG,R,DBT,CC,"HOLOGIC, Inc.",Selenia Dimensions,MG,2018-11-07,NaN,01:46:46,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.599610,1.618870,R CC Breast Tomosynthesis Image,73200000.0,1.232529,6.0,DERIVED\PRIMARY\TOMOSYNTHESIS\NONE,1.0,NaN,NaN,70.0,2457.0,1996.0,0.107029\0.107029
136002,4339601084,F,1972-07-01,46,65758752,DIAG,R,DBT,MLO,"HOLOGIC, Inc.",Selenia Dimensions,MG,2019-05-16,NaN,09:47:09,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,1.297690,1.151225,R MLO Breast Tomosynthesis Image,73200000.0,1.626641,60.0,DERIVED\PRIMARY\TOMOSYNTHESIS\NONE,1.0,NaN,NaN,69.0,2457.0,1996.0,0.107180\0.107180


In [40]:
dicom[dicom["series"]=="DBT"]["PATIENT_STUDY_ID"].unique().size

582

In [42]:
dicom[dicom['PATIENT_STUDY_ID']==4330066079]

,PATIENT_STUDY_ID,PatientSex,BIRTH_DATE,PatientAge,ACCESSION_NUMBER,study,side,series,view,Manufacturer,ManufacturerModelName,Modality,StudyDate,AcquisitionDate,StudyTime,StudyDescription,StudyInstanceUID,StudyID,SeriesDescription,SeriesNumber,SeriesInstanceUID,InstanceNumber,ImageType,SliceThickness,SpacingBetweenSlices,Exposure,NumberOfFrames,Rows,Columns,PixelSpacing
43,4330066079,F,1989-07-01,30,61259016,DIAG,NaN,NaN,NaN,"R2 Technology, Inc.",Cenova,SR,2020-01-14,NaN,18:25:58,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.215642,1.198432,NaN,3.0,1.225842,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
44,4330066079,F,1989-07-01,30,61259016,DIAG,R,C VIEW,MLO,"HOLOGIC, Inc.",Selenia Dimensions,MG,2020-01-14,NaN,18:25:57,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.215642,1.198432,R MLO C-View,71300000.0,1.279295,209.0,DERIVED\PRIMARY,NaN,NaN,57.0,NaN,2457.0,1890.0,0.088500\0.088500
45,4330066079,F,1989-07-01,30,61259016,DIAG,L,C VIEW,MLO,"HOLOGIC, Inc.",Selenia Dimensions,MG,2020-01-14,NaN,18:25:57,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.215642,1.198432,L MLO C-View,71300000.0,1.550463,215.0,DERIVED\PRIMARY,NaN,NaN,59.0,NaN,2457.0,1890.0,0.088500\0.088500
46,4330066079,F,1989-07-01,30,61259016,DIAG,L,C VIEW,CC,"HOLOGIC, Inc.",Selenia Dimensions,MG,2020-01-14,NaN,18:25:57,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.215642,1.198432,L CC C-View,71300000.0,1.145388,221.0,DERIVED\PRIMARY,NaN,NaN,65.0,NaN,2457.0,1890.0,0.088227\0.088227
47,4330066079,F,1989-07-01,30,61259016,DIAG,R,C VIEW,CC,"HOLOGIC, Inc.",Selenia Dimensions,MG,2020-01-14,NaN,18:25:57,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.215642,1.198432,R CC C-View,71300000.0,1.988508,203.0,DERIVED\PRIMARY,NaN,NaN,65.0,NaN,2457.0,1890.0,0.088500\0.088500
48,4330066079,F,1989-07-01,30,61259016,DIAG,L,DBT,MLO,"HOLOGIC, Inc.",Selenia Dimensions,MG,2020-01-14,NaN,18:25:57,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.215642,1.198432,L MLO Breast Tomosynthesis Image,73200000.0,1.256929,213.0,DERIVED\PRIMARY\VOLUME\NONE,1.0,NaN,NaN,47.0,2457.0,1890.0,0.088500\0.088500
49,4330066079,F,1989-07-01,30,61259016,DIAG,L,DBT,CC,"HOLOGIC, Inc.",Selenia Dimensions,MG,2020-01-14,NaN,18:25:57,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.215642,1.198432,L CC Breast Tomosynthesis Image,73200000.0,1.338235,219.0,DERIVED\PRIMARY\VOLUME\NONE,1.0,NaN,NaN,49.0,2457.0,1890.0,0.088364\0.088364
50,4330066079,F,1989-07-01,30,61259016,DIAG,R,DBT,MLO,"HOLOGIC, Inc.",Selenia Dimensions,MG,2020-01-14,NaN,18:25:58,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.215642,1.198432,R MLO Breast Tomosynthesis Image,73200000.0,1.301539,207.0,DERIVED\PRIMARY\VOLUME\NONE,1.0,NaN,NaN,47.0,2457.0,1890.0,0.088500\0.088500
51,4330066079,F,1989-07-01,30,61259016,DIAG,R,DBT,CC,"HOLOGIC, Inc.",Selenia Dimensions,MG,2020-01-14,NaN,18:25:58,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,1.215642,1.198432,R CC Breast Tomosynthesis Image,73200000.0,1.185590,201.0,DERIVED\PRIMARY\VOLUME\NONE,1.0,NaN,NaN,46.0,2457.0,1890.0,0.088500\0.088500
